<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB04_NumPy_for_Engineering_Computation_Vectors_Matrices_and_Linear_Algebra_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB04 · Clase 4 — NumPy para cálculo en ingeniería: vectores, matrices y álgebra lineal**

## Bloque 2: IA — Machine Learning (continuación)

`NB03` cubrió la mecánica de un único array. Esta clase cubre lo que ocurre **entre** arrays: las reglas de broadcasting que permiten a NumPy combinar arrays de formas distintas sin escribir un bucle, operaciones reales de matrices/vectores (`dot`, `@`, `einsum`), y `numpy.linalg` para resolver sistemas reales de ecuaciones lineales — aplicado a un problema real de estática naval: **encontrar la tensión en las líneas de amarre de un buque**. Esta es también el álgebra lineal en la que se apoya silenciosamente el PCA de `NB09` (autovectores de una matriz de covarianza), que aquí se hace explícita en vez de quedar oculta dentro de `sklearn`.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar las reglas de broadcasting de NumPy y usarlas para evitar bucles explícitos sobre datos reales.
- Cambiar de forma (reshape) y apilar arrays para ajustarlos a la forma que necesita un cálculo.
- Calcular productos escalares, multiplicación de matrices y productos externos, y saber cuándo aplica cada uno.
- Plantear y resolver un sistema real de ecuaciones lineales con `numpy.linalg.solve`.
- Calcular autovalores/autovectores y conectarlos, de forma concreta, con lo que hace el PCA por dentro.

### Agenda (clase de 2 horas)

| # | Sección | Minutos |
|---|---|---|
| 1 | Repaso y por qué importa | 5 |
| 2 | Reshape y apilado | 15 |
| 3 | Broadcasting | 20 |
| 4 | Operaciones vectoriales y matriciales (dot, `@`, outer, `einsum`, norma) | 20 |
| 5 | Resolver un problema real de estática naval con `linalg.solve` | 30 |
| 6 | Autovalores/autovectores: qué hace el PCA por dentro | 15 |
| 7 | Resumen, tarea, próxima clase | 15 |

Como siempre: orientación aproximada, no un guion cerrado.

---

## 1. Por qué importa

`NB07`–`NB20` se apoyan constantemente en operaciones *entre* arrays, no solo dentro de uno: combinar un offset de calibración por sensor con una matriz completa de lecturas, calcular una suma ponderada de características, o resolver un problema real de equilibrio físico. `Esta clase hace esas operaciones explícitas y correctas, en vez de confiar en que "simplemente funcionan" dentro de una llamada a una librería`.

---

## 2. Reshape y apilado

`reshape` `cambia la forma de un array sin cambiar sus datos ni (normalmente) su memoria` — es una vista siempre que es posible, siguiendo la regla de la Sección 5 de `NB03`. `hstack`/`vstack`/`concatenate` combinan arrays separados en uno solo, en direcciones distintas.

In [ ]:
import numpy as np

readings_flat = np.arange(12.0)   # imagine 12 real sensor readings, logged one after another
print("flat:", readings_flat)

readings_grid = readings_flat.reshape(3, 4)   # 3 sensors x 4 timestamps
print("\nreshaped to (3 sensors, 4 timestamps):\n", readings_grid)

print("\nShares memory with the original?", np.shares_memory(readings_flat, readings_grid))


Ahora dos secuencias reales de sensores independientes, combinadas de dos formas distintas.

In [ ]:
sensor_a = np.array([12.1, 12.4, 12.3])
sensor_b = np.array([15.0, 15.2, 14.9])

stacked_rows = np.vstack([sensor_a, sensor_b])       # stacked as two rows
stacked_cols = np.hstack([sensor_a, sensor_b])       # concatenated end to end

print("vstack (2 sensors x 3 readings):\n", stacked_rows)
print("\nhstack (one long sequence of 6 readings):", stacked_cols)


**Pruébalo tú mismo**: apila `sensor_a`, `sensor_b` y una tercera secuencia `sensor_c = np.array([11.8, 12.0, 11.9])` en un único array `(3, 3)` con `np.vstack`, y después usa una máscara booleana para encontrar todas las lecturas individuales por encima de `13.0` en los tres sensores a la vez (pista: una simple comparación sobre todo el array 2-D funciona, porque NumPy hace broadcasting de un escalar contra cualquier forma).

In [ ]:
sensor_c = np.array([11.8, 12.0, 11.9])
all_sensors = np.vstack([sensor_a, sensor_b, sensor_c])

print("Stacked shape:", all_sensors.shape)
print("Readings above 13.0:", all_sensors[all_sensors > 13.0])


> **Para saber más**: [documentación de `numpy.reshape`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) | [rutinas de manipulación de arrays de NumPy](https://numpy.org/doc/stable/reference/routines.array-manipulation.html)

---

## 3. Broadcasting

**Broadcasting** es la regla que usa NumPy para combinar arrays de formas *distintas* sin un bucle explícito, "estirando" mentalmente el array más pequeño sobre el más grande — `siempre que sus formas sean compatibles desde la dimensión final hacia dentro` (iguales, o que una de ellas sea `1`). Así es exactamente como el escalado de características de `NB07` y la aritmética de rejilla de `NB18` evitan escribir un bucle `for` de Python sobre cada fila.

Un ejemplo real: tres sensores, cada uno con su propio offset de calibración fijo, registrados en 4 instantes de tiempo. Restar un array (3,) de offsets a un array (3, 4) de lecturas hace broadcasting del offset automáticamente sobre cada instante.

In [ ]:
readings = np.array([
    [20.1, 20.3, 20.0, 20.4],   # sensor 0, 4 timestamps
    [19.8, 19.9, 20.1, 19.7],   # sensor 1
    [21.0, 21.2, 20.9, 21.1],   # sensor 2
])
calibration_offset = np.array([0.1, -0.2, 0.05])   # one offset per sensor

corrected = readings - calibration_offset[:, np.newaxis]   # shape (3,) -> (3, 1) to broadcast against (3, 4)

print("Raw readings:\n", readings)
print("\nCorrected readings:\n", corrected)


`calibration_offset[:, np.newaxis]` cambia la forma del offset de `(3,)` a `(3, 1)` — sin esto, NumPy intentaría hacer broadcasting de `(3,)` directamente contra la *última* dimensión de `(3, 4)` (4), que no coincide con 3, y lanzaría un `ValueError` real. La siguiente celda muestra ese error a propósito, porque saber reconocerlo es una habilidad realmente útil.

In [ ]:
try:
    broken = readings - calibration_offset   # (3, 4) vs (3,) -- broadcasts against the WRONG axis
except ValueError as e:
    print(f"ValueError (expected): {e}")


**Pruébalo tú mismo**: el broadcasting no se limita a ajustar la forma de un array a la de otro — dos arrays 1-D con *orientaciones* distintas pueden combinarse en una rejilla 2-D completa sin bucles ni apilado explícito. Predice la forma resultante y compruébalo: ¿qué produce `np.array([10, 20, 30])[:, np.newaxis] + np.array([1, 2, 3, 4])`?

In [ ]:
row_offsets = np.array([10, 20, 30])[:, np.newaxis]   # shape (3, 1)
col_offsets = np.array([1, 2, 3, 4])                  # shape (4,)
addition_table = row_offsets + col_offsets            # broadcasts to (3, 4)

print("Shape:", addition_table.shape)
print(addition_table)


> **Para saber más**: [documentación de broadcasting de NumPy](https://numpy.org/doc/stable/user/basics.broadcasting.html)

---

## 4. Operaciones vectoriales y matriciales

Tres operaciones reales, cada una respondiendo a una pregunta distinta:
- **Producto escalar** (`np.dot`, o `@` para matrices): un único número que resume cómo se alinean dos vectores — la base de toda suma ponderada, incluido el `weights @ inputs` de una red neuronal a partir de `NB11`.
- **Producto externo** (`np.outer`): todos los productos por pares entre los elementos de dos vectores, produciendo una matriz completa.
- **`einsum`**: una notación compacta y explícita de exactamente qué índices se suman — más útil en cuanto una operación es más compleja que un simple producto escalar o una multiplicación de matrices.

(El código NumPy antiguo a veces usa una clase específica `np.matrix` para álgebra lineal 2-D — la propia documentación de NumPy ahora desaconseja su uso a favor de un `ndarray` normal con el operador `@`, que es lo que usa esta clase en todo momento.)

In [ ]:
weights = np.array([0.5, 0.3, 0.2])
features = np.array([10.0, 20.0, 5.0])

weighted_sum = np.dot(weights, features)   # a single number: 0.5*10 + 0.3*20 + 0.2*5
print("Dot product (weighted sum):", weighted_sum)

A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[5.0, 6.0], [7.0, 8.0]])
print("\nMatrix product A @ B:\n", A @ B)

outer = np.outer(weights, features)
print("\nOuter product (every pairwise product):\n", outer)

einsum_dot = np.einsum("i,i->", weights, features)   # same as np.dot(weights, features)
print("\neinsum reproduces the same dot product:", einsum_dot)


Una magnitud real más, necesaria con frecuencia: la **norma** de un vector (su magnitud/longitud) — `np.linalg.norm`. Para el problema de las líneas de amarre de la Sección 5, así es exactamente como comprobarías si una tensión resuelta es irrazonablemente grande para la resistencia nominal de una línea real.

In [ ]:
force_vector = np.array([30.0, -40.0])   # a 2-D force, in kN, along x and y
magnitude = np.linalg.norm(force_vector)
print(f"Force components: {force_vector} kN")
print(f"Magnitude (norm): {magnitude:.2f} kN")

# Consistent with the classic 3-4-5 triangle, scaled by 10:
print("Matches sqrt(30^2 + 40^2)?", np.isclose(magnitude, np.sqrt(30.0**2 + 40.0**2)))


> **Para saber más**: [documentación de `numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html) | [Norma (matemáticas) (Wikipedia)](https://en.wikipedia.org/wiki/Norm_%28mathematics%29)

> **Para saber más**: [Producto escalar (Wikipedia)](https://en.wikipedia.org/wiki/Dot_product) | [documentación de `numpy.einsum`](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html)

---

## 5. Resolver un problema real de estática naval con `linalg.solve`

Un buque amarrado junto a un muelle se mantiene sujeto por dos líneas de amarre en ángulos fijos, resistiendo una fuerza combinada de viento y corriente. `Para que el buque permanezca en equilibrio estático, la suma de las componentes x e y de todas las fuerzas debe ser exactamente cero`. Eso da **dos ecuaciones con dos incógnitas** — las dos tensiones de línea `T1` y `T2` — un sistema real de ecuaciones lineales, exactamente la forma para la que está pensado `numpy.linalg.solve`: `A @ T = b`.

Planteamiento: la línea 1 tira a 30° de la línea central del buque, la línea 2 a 150° (direcciones aproximadamente opuestas, una configuración realista de esprines de proa y popa), y la fuerza ambiental es de 40 kN a 200° (empujando al buque fuera del muelle).

![Vista en planta de un buque amarrado a un muelle con dos líneas de amarre, T1 a 30 grados y T2 a 150 grados respecto a la crujía, y una fuerza ambiental de 40 kilonewtons a 200 grados empujando el buque fuera del muelle.](https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/images/nb04_mooring_diagram_es.svg)

*Vista en planta del buque amarrado: las líneas **T1** (30°) y **T2** (150°) tiran hacia el muelle desde proa y popa, mientras la fuerza ambiental **F** (40 kN a 200°) empuja el buque en sentido contrario, hacia fuera del muelle. Ángulos medidos en sentido antihorario desde la proa (0°), igual que en el código de la celda siguiente.*

In [ ]:
import numpy as np

angle1_deg, angle2_deg = 30.0, 150.0
env_force_kN, env_angle_deg = 40.0, 200.0

a1, a2, ae = np.radians([angle1_deg, angle2_deg, env_angle_deg])

# Equilibrium: T1 * (cos a1, sin a1) + T2 * (cos a2, sin a2) + F * (cos ae, sin ae) = 0
A = np.array([
    [np.cos(a1), np.cos(a2)],
    [np.sin(a1), np.sin(a2)],
])
b = np.array([
    -env_force_kN * np.cos(ae),
    -env_force_kN * np.sin(ae),
])

T1, T2 = np.linalg.solve(A, b)
print(f"Line 1 tension: {T1:.2f} kN")
print(f"Line 2 tension: {T2:.2f} kN")


**Verifica siempre un sistema resuelto contra las ecuaciones originales** — `linalg.solve` devolverá *un* número incluso si el planteamiento en sí fuera erróneo, así que comprobar el residuo es disciplina de ingeniería real, no opcional.

In [ ]:
residual = A @ np.array([T1, T2]) - b
print("Residual (should be ~0):", residual)
print("Max absolute residual:", np.abs(residual).max())

det_A = np.linalg.det(A)
print(f"\ndet(A) = {det_A:.4f} -- far from zero, so this system has exactly one solution")
print("(a near-zero determinant would mean the two lines pull in almost the same direction,")
print(" making the system poorly conditioned -- physically, an unsafe mooring arrangement)")


**Leyendo el resultado real con honestidad**: resolver este sistema da una tensión *negativa* para una de las dos líneas. Una línea de amarre real es un cabo — solo puede tirar, nunca empujar, así que una "tensión" negativa no es físicamente alcanzable; esa línea simplemente quedaría floja, y el buque necesitaría apoyarse en defensas o en una geometría de líneas distinta para resistir la fuerza restante. Esto no es un error en las matemáticas: es `linalg.solve` informando correctamente de que *esta idealización exacta* de dos líneas y dos incógnitas no tiene solución físicamente realizable para estos ángulos concretos y esta fuerza ambiental — un hallazgo de ingeniería real y útil, no un fallo que haya que justificar.

Una disposición de amarre real suele tener *más* líneas de las que las 2 ecuaciones de equilibrio pueden fijar de forma única (redundancia por seguridad, y exactamente el tipo de posibilidad de línea floja que acabamos de ver). Con 3 o más tensiones de línea pero todavía solo 2 ecuaciones de equilibrio independientes en esta simplificación 2-D, el sistema está **indeterminado**, no sobredeterminado: en vez de cero soluciones exactas, ahora hay infinitas combinaciones de tensiones que satisfacen el equilibrio exactamente. `numpy.linalg.lstsq` puede resolverlo igualmente, devolviendo la solución de norma mínima dentro de esa familia infinita. Esa extensión se deja como tarea de esta clase.

> **Para saber más**: [Sistema de ecuaciones lineales (Wikipedia)](https://en.wikipedia.org/wiki/System_of_linear_equations) | [documentación de `numpy.linalg.solve`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html) | [Amarre (embarcaciones) (Wikipedia)](https://en.wikipedia.org/wiki/Mooring_%28watercraft%29)

**Pruébalo tú mismo**: el ángulo de la fuerza ambiental (200°) fue lo que produjo una tensión negativa, físicamente imposible. Prueba con `env_angle_deg = 250.0` en su lugar (manteniendo todo lo demás igual) — ¿salen esta vez ambas tensiones positivas?

In [ ]:
env_angle_deg_2 = 250.0
ae2 = np.radians(env_angle_deg_2)

b2 = np.array([
    -env_force_kN * np.cos(ae2),
    -env_force_kN * np.sin(ae2),
])
T1_b, T2_b = np.linalg.solve(A, b2)
print(f"At 250 degrees -- Line 1 tension: {T1_b:.2f} kN, Line 2 tension: {T2_b:.2f} kN")
print("Both physically realizable (positive)?", T1_b > 0 and T2_b > 0)


---

## 6. Autovalores/autovectores: qué hace el PCA por dentro

El PCA de `NB09` redujo el dataset real de Sonar, de 60 dimensiones, a 2 dimensiones usando `sklearn.decomposition.PCA`, sin mostrar qué ocurre dentro de esa llamada. La operación central es: calcular la **matriz de covarianza** de los datos, y después encontrar los **autovectores** de esa matriz (las direcciones de máxima dispersión) y sus **autovalores** (cuánta dispersión hay a lo largo de cada uno). Esta clase lo calcula manualmente, sobre dos bandas de frecuencia reales y correlacionadas del dataset Sonar, `y lo confirma a mano en vez de confiar únicamente en la librería`.

In [ ]:
import urllib.request

sonar_url = "https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data"
raw_lines = urllib.request.urlopen(sonar_url).read().decode("utf-8").strip().split("\n")
sonar_features = np.array([[float(x) for x in line.split(",")[:-1]] for line in raw_lines])

two_bands = sonar_features[:, [10, 11]]   # two real, adjacent frequency bands -- likely correlated
print("Correlation between these two real bands:", np.corrcoef(two_bands.T)[0, 1].round(3))

cov_matrix = np.cov(two_bands.T)
print("\nCovariance matrix:\n", cov_matrix)

eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
print("\nEigenvalues:", eigenvalues)
print("Eigenvectors (columns):\n", eigenvectors)

dominant = eigenvectors[:, np.argmax(eigenvalues)]
print(f"\nDominant direction (largest eigenvalue): {dominant}")
print("This is exactly the direction sklearn's PCA would call 'the first principal component'.")


Ver los autovectores directamente sobre los datos reales hace que "dirección de máxima dispersión" sea algo concreto en vez de abstracto:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(two_bands[:, 0], two_bands[:, 1], s=15, alpha=0.5, color="steelblue")

mean_point = two_bands.mean(axis=0)
for i in range(2):
    # eig() returns complex dtype even for this real, symmetric covariance matrix --
    # .real is safe here since the imaginary part is mathematically guaranteed to be 0
    vec = (eigenvectors[:, i] * np.sqrt(eigenvalues[i]) * 3).real
    ax.annotate("", xy=mean_point + vec, xytext=mean_point,
                arrowprops=dict(arrowstyle="->", color="darkred", linewidth=2))

ax.set_xlabel("Band 10")
ax.set_ylabel("Band 11")
ax.set_title("Real Sonar data with both eigenvector directions overlaid")
ax.axis("equal")
plt.show()


> **Para saber más**: [Autovalores y autovectores (Wikipedia)](https://en.wikipedia.org/wiki/Eigenvalues_and_eigenvectors) | [documentación de `numpy.linalg.eig`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eig.html) | [Matriz de covarianza (Wikipedia)](https://en.wikipedia.org/wiki/Covariance_matrix)

---

## Resumen de la clase

- El reshape y el apilado reorganizan los datos en la forma que realmente necesita un cálculo, normalmente sin copiarlos (según la regla de vistas de `NB03`).
- El broadcasting combina arrays de formas compatibles-pero-distintas sin un bucle explícito — realmente útil, y una fuente real de `ValueError` cuando las formas no encajan como se pretendía.
- `dot`/`@`/outer/`einsum` son herramientas distintas para preguntas distintas: una única suma ponderada, un producto matricial completo, o todos los productos por pares.
- Un sistema real de ecuaciones lineales (tensiones de líneas de amarre, en equilibrio) se resuelve directamente con `numpy.linalg.solve`, y se comprueba con el residuo y el determinante — resolver no es lo mismo que verificar.
- Los autovectores de una matriz de covarianza son exactamente lo que calcula el PCA (`NB09`) por dentro — ya no es una caja negra.

## Para la próxima clase (NB05)

Una clase sistemática sobre Matplotlib: el modelo de objetos Figure/Axes, el estilo, las disposiciones de subplots y los gráficos con mapas de color — la base de representación gráfica que todos los notebooks de este curso han estado usando de forma improvisada desde `NB02`.

## Tarea / Ideas de práctica

1. Extiende el problema de amarre de la Sección 5 a 3 líneas (3 incógnitas) añadiendo una tercera línea y una ecuación de equilibrio de momentos respecto al centro de gravedad del buque.
2. Haz que el sistema de la Sección 5 sea singular a propósito (fija los dos ángulos de línea iguales) y observa qué hacen `np.linalg.det` y `np.linalg.solve` — lee el mensaje de error real.
3. Investiga `numpy.linalg.lstsq` y úsalo para resolver un sistema de amarre de 3 líneas **indeterminado** (3 incógnitas, pero solo 2 ecuaciones de equilibrio independientes usando la simplificación 2-D de este curso) — con más incógnitas que ecuaciones, ¿qué representa realmente aquí la solución que devuelve `lstsq` (pista: no es un mejor ajuste a datos con ruido, sino la solución de norma mínima entre infinitas soluciones exactas)?
4. Elige dos bandas de frecuencia del dataset Sonar que *no* estén correlacionadas (compruébalo antes con `np.corrcoef`) y repite el cálculo de autovectores de la Sección 6 — ¿en qué se diferencia el resultado del par correlacionado usado en clase?
5. Haz broadcasting de un array (4,) de factores de corrección por instante de tiempo (no por sensor) contra el array `readings` de la Sección 3 — ¿qué forma de reshape necesita `[np.newaxis, :]` aquí, comparado con el caso por sensor mostrado en clase?

> ***Como siempre: resolver una ecuación real es solo la mitad del trabajo — comprobar el residuo es la otra mitad.***